# 03 - Baseline Modeling and AutoML Benchmarks

Benchmark strong non-deep baselines across all forecast horizons.


## Why Baselines Matter

**Definition**  
Baselines define minimum acceptable performance and protect against over-engineering.

**Theory**  
If advanced models do not beat robust baselines out-of-sample, complexity is unjustified.

**Mathematical Intuition**  
Error gap vs baseline quantifies real incremental value.

**Financial Intuition**  
In finance, simpler models can outperform during regime stability due lower variance.

**Business Impact**  
Baseline discipline reduces deployment risk and accelerates debugging.

**Real-World Example**  
Naive model surprisingly strong on short horizons in persistent trends.

**Visual Explanation**  
Code cells below generate real plots from Apple market data.

**Code Explanation**  
Each code block is annotated inline and uses shared production modules from `src/`.

**Interpretation of Results**  
After each output, interpret what signal quality, risk, and forecasting reliability imply.


In [ ]:

import sys
import os
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.forecast_pipeline import ForecastingFramework
from src.data_loader import load_stock_data, split_data
from src.features import create_features
from src.evaluation import regression_metrics
from src.visualization import *

OUT = PROJECT_ROOT / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'
FAST_NOTEBOOK_MODE = os.getenv('NOTEBOOK_FULL_RUN', '0') != '1'

def make_framework():
    framework = ForecastingFramework(str(CONFIG_PATH))
    if FAST_NOTEBOOK_MODE:
        framework.config['models']['automl']['lazypredict'] = False
        framework.config['models']['automl']['pycaret'] = False
        framework.config['models']['automl']['flaml'] = False
        framework.config['models']['deep_learning']['epochs'] = 8
        framework.config['models']['deep_learning']['early_stopping_patience'] = 3
        framework.config['weight_optimization']['methods'] = ['grid']
    return framework

framework = make_framework()


In [ ]:

framework = make_framework()
framework.load_data()

horizons = [1] if FAST_NOTEBOOK_MODE else framework.config['features']['horizons']
all_rows = []
automl_rows = []

for h in horizons:
    print(f'Running baseline benchmark for horizon {h}...')
    out = framework.train_baselines(h)
    lb = out['leaderboard'].copy()
    lb['horizon'] = h
    all_rows.append(lb)

    for tool_name, tool_df in out['automl'].items():
        tmp = tool_df.copy()
        tmp['horizon'] = h
        tmp['tool_name'] = tool_name
        automl_rows.append(tmp)

baseline_all = pd.concat(all_rows, ignore_index=True)
baseline_all.to_csv(OUT / 'tables/03_baseline_all_horizons.csv', index=False)
display(baseline_all.head(20))

if automl_rows:
    automl_all = pd.concat(automl_rows, ignore_index=True)
    automl_all.to_csv(OUT / 'tables/03_automl_all_horizons.csv', index=False)
    display(automl_all.head(20))


`NOTEBOOK_FULL_RUN=1` executes all configured horizons; default mode runs horizon 1 for faster reproducible execution.


## Tool Tradeoffs Summary

| Tool | Strength | Weakness | Practical Use |
|---|---|---|---|
| LazyPredict | Fast breadth | Limited tuning depth | Rapid first scan |
| PyCaret | Strong experiment workflow | Heavier abstraction/runtime | Structured comparison tables |
| FLAML | Budget-aware optimization | Less tutorial-style UX | Time-constrained tuning |
